# 生日五行金融密碼計算器

## 1. 環境設定

本機 Jupyter 請先在專案資料夾執行 `uv sync`，再使用 JupyterLab 或 Jupyter Notebook 開啟本檔案並由上到下執行 cells。


## 2. 載入套件與設定


In [1]:
from datetime import date, datetime
import math

import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Circle, Patch
from IPython.display import HTML, display


def configure_matplotlib_font():
    """設定 Matplotlib 中文字型，提升本機 Jupyter 顯示穩定度。"""
    preferred_fonts = [
        "Noto Sans CJK TC",
        "Noto Sans CJK JP",
        "Noto Sans CJK SC",
        "Microsoft JhengHei",
        "PingFang TC",
        "Heiti TC",
        "SimHei",
        "WenQuanYi Zen Hei",
        "Arial Unicode MS",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in preferred_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.family"] = font_name
            break
    plt.rcParams["axes.unicode_minus"] = False


configure_matplotlib_font()


## 3. 計算邏輯


In [2]:
WUXING_BY_DIGIT = {
    1: "木",
    6: "木",
    2: "火",
    7: "火",
    5: "土",
    0: "土",
    4: "金",
    9: "金",
    3: "水",
    8: "水",
}

WUXING_COLORS = {
    "木": "#2E8B57",
    "火": "#E4572E",
    "土": "#B08D57",
    "金": "#C9A227",
    "水": "#2F80ED",
}


def split_digits(number):
    """將整數拆成各位數字清單。"""
    return [int(digit) for digit in str(number)]


def parse_birthday(value):
    """解析 date 物件，並保留生日字串格式支援以利測試與重用。"""
    if isinstance(value, date):
        birthday = value
    else:
        value = str(value).strip()
        supported_formats = ("%Y-%m-%d", "%Y%m%d", "%Y/%m/%d")
        for birthday_format in supported_formats:
            try:
                birthday = datetime.strptime(value, birthday_format).date()
                break
            except ValueError:
                continue
        else:
            raise ValueError("請輸入有效日期，格式可為 2000-01-01、20000101 或 2000/01/01。")

    if birthday > date.today():
        raise ValueError("出生日期不可晚於今天。")

    return birthday


def calculate_birthday_password(birthday):
    """計算生日數字密碼並保留每次拆位加總過程。"""
    birthday = parse_birthday(birthday)
    birthday_digits = [int(digit) for digit in birthday.strftime("%Y%m%d")]
    steps = []
    current_digits = birthday_digits

    while True:
        total = sum(current_digits)
        steps.append((current_digits, total))
        if total < 10:
            return total, steps
        current_digits = split_digits(total)


def format_steps(steps):
    """將生日密碼計算步驟格式化成可在 Notebook 顯示的 HTML 文字。"""
    lines = []
    for index, (digits, total) in enumerate(steps, start=1):
        expression = " + ".join(str(digit) for digit in digits)
        lines.append(f"第 {index} 次：{expression} = {total}")
    return "<br>".join(lines)


## 4. Matplotlib 河圖五行圖


In [3]:
def create_wuxing_figure(selected_digit=None):
    """建立 Matplotlib 河圖五行數字對應圖。"""
    digits = list(range(10))
    radius = 2.3
    digit_points = []

    for index, digit in enumerate(digits):
        angle = math.pi / 2 - (2 * math.pi * index / len(digits))
        element = WUXING_BY_DIGIT[digit]
        digit_points.append(
            {
                "digit": digit,
                "element": element,
                "x": radius * math.cos(angle),
                "y": radius * math.sin(angle),
                "color": WUXING_COLORS[element],
            }
        )

    element_order = ["木", "火", "土", "金", "水"]
    element_points = []
    for index, element in enumerate(element_order):
        angle = math.pi / 2 - (2 * math.pi * index / len(element_order))
        element_points.append(
            {
                "element": element,
                "x": 0.85 * math.cos(angle),
                "y": 0.85 * math.sin(angle),
                "color": WUXING_COLORS[element],
                "digits": ", ".join(str(d) for d in digits if WUXING_BY_DIGIT[d] == element),
            }
        )

    fig, ax = plt.subplots(figsize=(8, 6.5), facecolor="#FAFAFA")
    ax.set_facecolor("#FAFAFA")

    for point in digit_points:
        element_point = next(item for item in element_points if item["element"] == point["element"])
        ax.plot(
            [element_point["x"], point["x"]],
            [element_point["y"], point["y"]],
            color=point["color"],
            linewidth=1.6,
            alpha=0.72,
            zorder=1,
        )

    for point in element_points:
        circle = Circle(
            (point["x"], point["y"]),
            radius=0.27,
            facecolor=point["color"],
            edgecolor="#333333",
            linewidth=1.2,
            zorder=3,
        )
        ax.add_patch(circle)
        ax.text(
            point["x"],
            point["y"],
            point["element"],
            ha="center",
            va="center",
            fontsize=18,
            color="white",
            zorder=4,
        )

    for point in digit_points:
        is_selected = point["digit"] == selected_digit
        circle = Circle(
            (point["x"], point["y"]),
            radius=0.32 if is_selected else 0.23,
            facecolor=point["color"],
            edgecolor="#111111" if is_selected else "#FFFFFF",
            linewidth=3.2 if is_selected else 1.8,
            zorder=5,
        )
        ax.add_patch(circle)
        ax.text(
            point["x"],
            point["y"],
            str(point["digit"]),
            ha="center",
            va="center",
            fontsize=19 if is_selected else 15,
            color="white",
            zorder=6,
        )

    title = "河圖五行數字對應圖"
    if selected_digit is not None:
        title += f"（已高亮：{selected_digit}）"

    ax.set_title(title, fontsize=18, pad=18)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-2.85, 2.85)
    ax.set_aspect("equal")
    ax.axis("off")
    legend_items = [
        Patch(facecolor=WUXING_COLORS[element], edgecolor="#333333", label=f"{element}: {', '.join(str(d) for d in digits if WUXING_BY_DIGIT[d] == element)}")
        for element in element_order
    ]
    ax.legend(
        handles=legend_items,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=5,
        frameon=False,
        fontsize=11,
    )
    fig.tight_layout()
    return fig


## 5. Jupyter Widget 操作介面

本區塊可在本機 JupyterLab 與 Jupyter Notebook 使用。請選擇出生日期，再按下「計算生日五行金融密碼」。


In [4]:
birthday_picker = widgets.DatePicker(
    description="出生日期",
    disabled=False,
    value=date(2000, 1, 1),
)

calculate_button = widgets.Button(
    description="計算生日五行金融密碼",
    button_style="primary",
    icon="calculator",
    layout=widgets.Layout(width="240px"),
)

result_output = widgets.Output()


def render_birthday_result(birthday):
    """顯示生日密碼、五行屬性與 Matplotlib 圖表。"""
    try:
        password, steps = calculate_birthday_password(birthday)
        birthday = parse_birthday(birthday)
    except ValueError as error:
        display(HTML(f"<p style='color:#B00020; font-weight:600;'>輸入錯誤：{error}</p>"))
        return

    element = WUXING_BY_DIGIT[password]

    display(
        HTML(
            f"""
            <div style="font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; line-height: 1.65;">
                <h2 style="margin-bottom: 0.4rem;">計算結果</h2>
                <p><strong>出生日期：</strong>{birthday.strftime('%Y-%m-%d')}</p>
                <p><strong>拆位加總流程：</strong><br>{format_steps(steps)}</p>
                <p><strong>生日數字密碼：</strong><span style="font-size: 1.4rem; font-weight: 700; color: #111;"> {password}</span></p>
                <p><strong>河圖五行屬性：</strong><span style="font-size: 1.2rem; font-weight: 700; color: {WUXING_COLORS[element]};"> {element}</span></p>
            </div>
            """
        )
    )
    fig = create_wuxing_figure(password)
    display(fig)
    plt.close(fig)


def render_widget_result(_button):
    """處理計算按鈕點擊事件並更新 Notebook 輸出區。"""
    with result_output:
        result_output.clear_output()

        birthday = birthday_picker.value
        if birthday is None:
            display(HTML("<p style='color:#B00020; font-weight:600;'>請先選擇出生日期。</p>"))
            return

        render_birthday_result(birthday)


calculate_button.on_click(render_widget_result)

display(widgets.VBox([widgets.HBox([birthday_picker, calculate_button]), result_output]))
